## Question 1: [IPO] Withdrawn IPOs by Company Type

**What is the total withdrawn IPO value (in $ millions) for the company class with the highest total withdrawal value?**

In [ ]:
import pandas as pd
from utils import download_ipo_data, calculate_average_price, map_types

In [2]:
df = download_ipo_data(url = 'https://stockanalysis.com/ipos/withdrawn/')
# df = pd.read_csv('ipo_withdrawn.csv')
df['Company Name'] = df['Company Name'].str.lower()
df

,Symbol,Company Name,Price Range,Shares Offered
0,ODTX,"odyssey therapeutics, inc.",-,-
1,UNFL,"unifoil holdings, inc.",$3.00 - $4.00,2000000
2,AURN,"aurion biotech, inc.",-,-
3,ROTR,"phi group, inc.",-,-
4,ONE,one power company,-,-
...,...,...,...,...
95,FHP,"freehold properties, inc.",-,-
96,CHO,chobani inc.,-,-
97,IFIT,ifit health & fitness inc.,$18.00 - $21.00,30769231
98,GLGX,"gerson lehrman group, inc.",-,-


In [3]:
# Create regex pattern from keys
pattern1 = r'(' + '|'.join(map_types.keys()) + r')\.?'
# Create new Company Class column
df['Company Class'] = df['Company Name'].str.extract(pattern1)
df['Company Class'] = df['Company Class'].map(map_types).fillna('Other')

- Some of the mappings are not correct, so I have to manually fix it.

In [4]:
# Map 'group inc' to 'Inc'
group_to_inc_idx = df[df['Company Name'].str.contains('group inc.')].index.to_list()
df.loc[group_to_inc_idx, 'Company Class'] = 'Inc'

# Map 'group acquisition corporation' to 'Acq.Corp'
df.loc[df[df['Company Name'].str.contains('group acquisition corporation')].index.to_list(), 'Company Class'] = 'Acq.Corp'

# Map 'holdings, inc' to 'Inc'
df.loc[df[df['Company Name'].str.contains('holdings, inc')].index.tolist(), 'Company Class'] = 'Inc'

# Map 'holdings limited' to 'Ltd'
df.loc[df[df['Company Name'].str.contains('holdings limited')].index.tolist(), 'Company Class'] = 'Ltd'

# Should I map 'holdings corp' to 'Other'? 
df.loc[df[df['Company Name'].str.contains('holdings corp')].index.tolist(), 'Company Class'] = 'Other'

# Should I map 'acquisition corp limited' to Ltd? No, leave it as is
# df.loc[df[df['Company Name'].str.contains('acquisition corp limited')].index.tolist(), 'Company Class'] = 'Ltd'

In [5]:
# Calculate average price and write result to new 'Avg. Price' column
df['Avg. Price'] = df['Price Range'].apply(calculate_average_price)

# Convert Shares Offered column into numeric
df['Shares Offered'] = pd.to_numeric(df['Shares Offered'], errors='coerce')
df['Shares Offered'] = df['Shares Offered'].fillna(0)

# Calculate withdrawn value
df['Withdrawn Value'] = df['Shares Offered'] * df['Avg. Price']

# Filter rows that are 0s in the Withdrawn Value column 
df_no_zeros =  df[~(df['Withdrawn Value'] == 0)] # 71 rows

In [6]:
df.groupby(by='Company Class')['Withdrawn Value'].sum().sort_values(ascending=False)

Company Class
Acq.Corp    4.021000e+09
Inc         1.949852e+09
Other       8.429200e+08
Ltd         5.497346e+08
Group       3.361000e+08
Holdings    5.000000e+06
Name: Withdrawn Value, dtype: float64

## Question 2: [IPO] Median Sharpe Ratio for 2024 IPOs (First 5 Months)


**What is the median Sharpe ratio (as of 6 June 2025) for companies that went public in the first 5 months of 2024?**



In [ ]:
import pandas as pd
from datetime import date
import numpy as np
from utils import download_ipo_data, fetch_ticker_data

In [2]:
ipo_2024 = download_ipo_data(url='https://stockanalysis.com/ipos/2024/')
ipo_2024

,IPO Date,Symbol,Company Name,IPO Price,Current,Return
0,"Dec 31, 2024",ONEG,OneConstruction Group Limited,$4.00,$3.40,-15.00%
1,"Dec 27, 2024",PHH,"Park Ha Biological Technology Co., Ltd.",$4.00,$20.48,412.00%
2,"Dec 23, 2024",HIT,"Health In Tech, Inc.",$4.00,$0.60,-85.00%
3,"Dec 23, 2024",TDAC,Translational Development Acquisition Corp.,$10.00,$10.26,2.58%
4,"Dec 20, 2024",RANG,Range Capital Acquisition Corp.,$10.00,$10.45,4.50%
...,...,...,...,...,...,...
220,"Jan 18, 2024",CCTG,CCSC Technology International Holdings Limited,$6.00,$1.08,-82.00%
221,"Jan 18, 2024",PSBD,Palmer Square Capital BDC Inc.,$16.45,$13.86,-15.75%
222,"Jan 12, 2024",SYNX,Silynxcom Ltd.,$4.00,$1.70,-57.50%
223,"Jan 11, 2024",SDHC,Smith Douglas Homes Corp.,$21.00,$19.25,-8.33%


In [7]:
# Filter to keep only those IPOs before 1 June 2024 
ipo_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   IPO Date      225 non-null    datetime64[ns]
 1   Symbol        225 non-null    object        
 2   Company Name  225 non-null    object        
 3   IPO Price     225 non-null    object        
 4   Current       225 non-null    object        
 5   Return        225 non-null    object        
dtypes: datetime64[ns](1), object(5)
memory usage: 10.7+ KB


In [3]:
# Change to datetime data type
ipo_2024['IPO Date'] = pd.to_datetime(ipo_2024['IPO Date'])
# Filter to keep only those IPOs before 1 June 2024 
ipo_first_5 = ipo_2024[ipo_2024['IPO Date'] < '2024-06-01']

# Filter rows without IPO Price
ipo_first_5 = ipo_first_5[~ipo_first_5['IPO Price'].str.contains('-')]

# Convert IPO Price column to numeric
ipo_first_5['IPO Price'] = pd.to_numeric(ipo_first_5['IPO Price'].str.replace(pat='$', repl='', regex=False))

In [8]:
# Download daily stock data for tickers that IPO-ed in the first 5 months of 2024
ticker_list = ipo_first_5['Symbol'].unique().tolist()

ticker_data = fetch_ticker_data(ticker_list=ticker_list)

In [ ]:
# # test BOW ticker
# ticker_obj = yf.Ticker(ticker_list[0])
# ticker_hist = ticker_obj.history(
#     period='max',
#     interval='1d'
# )

# # Create features from df downloaded from yf

# ticker_hist['Ticker'] = ticker_list[0]
# ticker_hist['Year'] = ticker_hist.index.year
# ticker_hist['Month'] = ticker_hist.index.month
# ticker_hist['Weekday'] = ticker_hist.index.weekday
# ticker_hist['Date'] = ticker_hist.index.date

# # ticker_hist

# # Examine historical returns for a range of trading days
# for i in [1,3,7,30,90,365]:
#     ticker_hist['growth_' + str(i) + 'd'] = ticker_hist['Close'] / ticker_hist['Close'].shift(i)

# ticker_hist['growth_future_30d'] = ticker_hist['Close'].shift(-30) / ticker_hist['Close']

# # ticker_hist

# # Examine 30d rolling volatility
# ticker_hist['volatility'] = ticker_hist['Close'].rolling(30).std() * np.sqrt(252)
# # ticker_hist

# # Examine historical returns for 252 trading days
# ticker_hist['growth_252d'] = ticker_hist['Close'] / ticker_hist['Close'].shift(252)
# ticker_hist['volatility'] = ticker_hist['Close'].rolling(252).std() * np.sqrt(252)
# ticker_hist['Sharpe'] = (ticker_hist['growth_252d'] - 0.045 /ticker_hist['volatility'])
# # ticker_hist

In [ ]:
# ticker_data.reset_index(inplace=True)
ticker_data['Date'] = pd.to_datetime(ticker_data['Date'])

In [ ]:
# Filter trading day June 6
ticker_data_subset = ticker_data[ticker_data['Date'] == '2025-06-06']
ticker_data_subset = ticker_data_subset[~ticker_data_subset['growth_252d'].isna()]
ticker_data_subset[['growth_252d', 'Sharpe']].describe()

,growth_252d,Sharpe
count,71.000000,71.000000
mean,1.152897,0.301597
std,1.406017,0.529685
min,0.024970,-0.079677
25%,0.293422,0.041215
50%,0.758065,0.083768
75%,1.362736,0.335681
max,8.097413,2.835668


## Question 3: [IPO] ‘Fixed Months Holding Strategy’
**What is the optimal number of months (1 to 12) to hold a newly IPO'd stock in order to maximize average growth?**

In [ ]:
# messed up, neex to drop those columns
cols_to_drop = ticker_data.columns[ticker_data.columns.str.startswith(('future', 'growth'))]
ticker_data = ticker_data.drop(columns=cols_to_drop)

In [83]:
ticker_data_copy = ticker_data.copy()
ticker_data_copy

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,volatility,Sharpe
0,2024-05-23,23.000000,24.270000,22.139999,23.799999,3335800,0.0,0.0,BOW,NaN,NaN
1,2024-05-24,24.260000,26.150000,23.980000,25.700001,990500,0.0,0.0,BOW,NaN,NaN
2,2024-05-28,25.850000,26.879999,25.075001,26.480000,555100,0.0,0.0,BOW,NaN,NaN
3,2024-05-29,26.440001,26.490000,25.500999,26.290001,302700,0.0,0.0,BOW,NaN,NaN
4,2024-05-30,27.209999,27.209999,25.500000,26.139999,200900,0.0,0.0,BOW,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
22866,2025-06-05,3.740000,4.135000,3.360000,3.570000,264600,0.0,0.0,ROMA,12.173744,0.501044
22867,2025-06-06,3.650000,3.950000,3.630000,3.700000,84400,0.0,0.0,ROMA,12.553202,0.486840
22868,2025-06-09,3.750000,3.955000,2.600000,2.860000,381700,0.0,0.0,ROMA,12.238182,0.374470
22869,2025-06-10,2.850000,3.310000,2.770000,2.770000,52500,0.0,0.0,ROMA,11.810360,0.450724


In [84]:
ticker_data_copy = ticker_data_copy[['Close', 'Ticker', 'Date']]
ticker_data_copy

,Close,Ticker,Date
0,23.799999,BOW,2024-05-23
1,25.700001,BOW,2024-05-24
2,26.480000,BOW,2024-05-28
3,26.290001,BOW,2024-05-29
4,26.139999,BOW,2024-05-30
...,...,...,...
22866,3.570000,ROMA,2025-06-05
22867,3.700000,ROMA,2025-06-06
22868,2.860000,ROMA,2025-06-09
22869,2.770000,ROMA,2025-06-10


In [ ]:
# historical returns
for i, m in enumerate([i for i in range(21,273, 21)]):
    ticker_data_copy.loc[:, 'future_growth_' + str(i+1) + 'm'] = ticker_data_copy['Close'].shift(-i) / ticker_data_copy['Close']
ticker_data_copy

In [86]:
# Determine the first trading day for each ticker
min_date = ticker_data_copy.sort_values('Date').groupby('Ticker').first().reset_index()
min_date

,Ticker,Close,Date,future_growth_1m,future_growth_2m,future_growth_3m,future_growth_4m,future_growth_5m,future_growth_6m,future_growth_7m,future_growth_8m,future_growth_9m,future_growth_10m,future_growth_11m,future_growth_12m
0,AHR,12.433781,2024-02-07,1.0,0.987897,0.992436,0.994705,1.009834,1.040847,1.031770,1.049168,1.014372,1.013616,1.024962,1.028744
1,ALAB,62.029999,2024-03-20,1.0,1.034177,1.128486,1.370305,1.344188,1.288409,1.196034,1.146542,1.164114,1.116879,1.124778,1.150089
2,ANRO,20.700001,2024-02-02,1.0,1.053140,0.936715,0.916908,0.916908,0.799517,0.672947,0.688406,0.724638,0.768116,0.766184,0.654589
3,AS,13.400000,2024-02-01,1.0,1.115672,1.089552,1.129105,1.123134,1.132836,1.114925,1.112687,1.082090,1.108209,1.156716,1.191791
4,AUNA,9.600000,2024-03-22,1.0,1.041667,1.097917,1.110417,1.109375,1.093750,1.029167,0.962500,0.979167,0.947917,0.833333,0.755208
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,YIBO,2.790000,2024-01-25,1.0,0.974910,0.931900,0.953405,0.867384,0.821505,0.737993,0.727599,0.788530,0.770609,0.770609,0.806452
71,YYGH,2.900000,2024-04-22,1.0,0.822414,0.779310,1.000000,1.013793,0.900000,0.813793,0.813793,0.765517,0.672414,0.693103,0.644828
72,ZBAO,3.700000,2024-04-02,1.0,0.891892,0.964865,0.918919,0.978378,0.962162,0.881081,0.913514,0.921622,0.935135,0.950000,0.964865
73,ZK,28.260000,2024-05-10,1.0,1.027601,0.977707,0.946214,0.941614,0.920028,0.959306,0.956829,0.941614,0.917197,0.827318,0.871550


In [ ]:
# Join the data
# future_growth = pd.merge(left=ticker_data_copy, right=min_date, how='inner', on=['Ticker', 'Date'])

In [ ]:
# summary_stats = min_date.describe()
# summary_stats.T.sort_values(by='mean', ascending=False)

In [95]:
# summary_stats_no_date = summary_stats.drop(columns='Date')
summary_stats_no_date.T.sort_values(by='mean', ascending=False)

,count,mean,min,25%,50%,75%,max,std
Close,75.0,15.350148,0.011600,4.140000,10.030000,19.424999,98.000000,17.869281
future_growth_4m,75.0,1.082155,0.197895,0.947839,1.000997,1.106717,2.938575,0.380276
future_growth_3m,75.0,1.065803,0.248421,0.951632,1.000498,1.093734,2.869779,0.335365
future_growth_5m,75.0,1.050385,0.152632,0.914594,1.002421,1.098010,2.825553,0.383363
future_growth_7m,75.0,1.046187,0.131001,0.909261,0.999003,1.082357,3.040299,0.453544
future_growth_8m,75.0,1.041265,0.119609,0.901996,1.000995,1.057339,4.070149,0.515895
future_growth_6m,75.0,1.040937,0.133684,0.912529,1.000000,1.080609,2.847666,0.410316
future_growth_10m,75.0,1.038534,0.109032,0.880019,1.000527,1.082064,3.432836,0.498847
future_growth_9m,75.0,1.033393,0.113914,0.891132,1.000995,1.062556,3.559702,0.481380
future_growth_12m,75.0,1.016377,0.089504,0.837312,1.000000,1.062814,3.549254,0.482965


## Question 4: : [Strategy] Simple RSI-Based Trading Strategy

**Apply a simple rule-based trading strategy using the Relative Strength Index (RSI) technical indicator to identify oversold signals and calculate profits.**

In [99]:
import gdown
import pandas as pd

In [101]:
# file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
# gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
df

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
0,0.054277,0.062259,0.054277,0.059598,1.031789e+09,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.059598,0.062791,0.059598,0.061726,3.081600e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.061726,0.063323,0.061726,0.062791,1.331712e+08,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.062791,0.063323,0.060662,0.061194,6.776640e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.061194,0.061726,0.059598,0.060130,4.789440e+07,0.0,0.0,MSFT,1986,1986-03-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5686,3615.800049,3672.500000,3608.399902,3648.699951,1.678934e+06,0.0,0.0,LT.NS,2025,2025-05-01,...,NaN,NaN,NaN,NaN,1.003714,1.020064,1.036306,1.156301,1.233323,1.597248
5687,3648.699951,3665.000000,3603.000000,3640.000000,2.013954e+06,0.0,0.0,LT.NS,2025,2025-05-01,...,0.993181,0.989654,0.781299,0.842957,0.995927,1.011165,1.020634,1.162549,1.292217,1.570651
5688,3660.000000,3663.000000,3620.000000,3646.300049,1.293244e+06,0.0,0.0,LT.NS,2025,2025-05-01,...,0.992203,1.000308,0.798376,0.886128,0.989061,0.988691,0.982898,1.135015,1.272691,1.578452
5689,3663.899902,3668.899902,3618.000000,3655.300049,1.972248e+06,0.0,0.0,LT.NS,2025,2025-05-01,...,0.978792,0.991959,0.794034,0.863857,0.979958,0.965291,0.945990,1.120454,1.252080,1.563254


In [105]:
rsi_threshold = 25
selected_df = df[
    (df['rsi'] < rsi_threshold) &
    (df['Date'] >= '2000-01-01') &
    (df['Date'] <= '2025-06-01')
]
selected_df

,Open,High,Low,Close_x,Volume,Dividends,Stock Splits,Ticker,Year,Month,...,growth_brent_oil_7d,growth_brent_oil_30d,growth_brent_oil_90d,growth_brent_oil_365d,growth_btc_usd_1d,growth_btc_usd_3d,growth_btc_usd_7d,growth_btc_usd_30d,growth_btc_usd_90d,growth_btc_usd_365d
3668,20.056772,20.114241,19.405453,19.673643,99915200.0,0.0,0.0,MSFT,2000,2000-09-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3669,19.692798,19.807736,19.060636,19.309669,69037800.0,0.0,0.0,MSFT,2000,2000-09-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3680,18.256067,18.332693,17.317403,17.336559,85374000.0,0.0,0.0,MSFT,2000,2000-10-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3681,17.279087,17.336557,16.704395,16.991741,136453400.0,0.0,0.0,MSFT,2000,2000-10-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3682,17.010902,17.547282,16.934277,16.972589,81099400.0,0.0,0.0,MSFT,2000,2000-10-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4405,797.393417,806.609179,764.362578,769.289795,7251950.0,0.0,0.0,LT.NS,2020,2020-03-01,...,0.764911,0.515014,0.457056,0.336604,1.181878,1.234663,1.245515,0.610451,0.857647,1.514674
4406,774.673304,800.951963,758.522860,788.998840,6130185.0,0.0,0.0,LT.NS,2020,2020-03-01,...,0.753842,0.491171,0.431611,0.320580,1.001225,1.186226,1.114145,0.643468,0.862000,1.538415
4407,738.175133,739.041975,644.192390,660.662170,7308612.0,0.0,0.0,LT.NS,2020,2020-03-01,...,0.813666,0.496236,0.434706,0.322131,1.100520,1.035093,1.279557,0.663996,0.876243,1.595238
4408,698.550405,699.244329,621.756122,654.971985,7110384.0,10.0,0.0,LT.NS,2020,2020-03-01,...,0.802068,0.509668,0.437480,0.319412,1.049637,1.088881,1.288802,0.678603,0.925726,1.699390


In [106]:
net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()
net_income

24295.523125248386